<a href="https://colab.research.google.com/github/Innovatewithapple/YoutubeSummarizationNLP/blob/main/YoutubeSummaryNLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yt-dlp openai-whisper

In [ ]:
import numpy as np
import random
import os
import torch

def setProductionSeed(isActive,seed=42):
  if isActive == True:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
      torch.cuda.manual_seed(seed)
      torch.cuda.manual_seed_all(seed)

      torch.backends.cudnn.deterministic = True
      torch.backends.cudnn.benchmark = False

setProductionSeed(True)

In [ ]:
import yt_dlp
import whisper
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer,AutoModelForCausalLM
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt_tab')
from torch.utils.data import DataLoader,Dataset

In [ ]:
from huggingface_hub import login
login(token="")

In [ ]:
#-----------Transcribe Model-----------!
transcribe_Model = whisper.load_model('small')

#--------Decoder-------!
decoder_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", trust_remote_code=True)
decoder_tokenizer.pad_token = decoder_tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct",dtype=torch.float16,device_map="auto",trust_remote_code=True)

In [ ]:
def download_audio(youtube_url, output_path="audio"):
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': f'{output_path}/%(title)s.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=True)
        filename = ydl.prepare_filename(info)
        # Return the actual .mp3 path
        return filename.rsplit('.', 1)[0] + '.mp3'

In [ ]:
audio_path = download_audio("https://www.youtube.com/watch?v=coX4duwUCpw")
print(f"Audio saved to: {audio_path}")

[youtube] Extracting URL: https://www.youtube.com/watch?v=coX4duwUCpw
[youtube] coX4duwUCpw: Downloading webpage


[youtube] coX4duwUCpw: Downloading android vr player API JSON
[info] coX4duwUCpw: Downloading 1 format(s): 251
[download] Destination: audio/Why New Smartphone Cameras Feel Worse.webm
[download] 100% of    6.59MiB in 00:00:00 at 33.50MiB/s  
[ExtractAudio] Destination: audio/Why New Smartphone Cameras Feel Worse.mp3
Deleting original file audio/Why New Smartphone Cameras Feel Worse.webm (pass -k to keep)
Audio saved to: audio/Why New Smartphone Cameras Feel Worse.mp3


In [ ]:
%%time
#--------Transcribe Audio----------!

transcribe_result = transcribe_Model.transcribe(audio_path)
transcript = transcribe_result['text']
print(f'Transcript: \n{transcript}')

Transcript: 
 Alright, so I decided to try something. I broke out every single iPhone generation from 1 to 17 and took the same picture with every single one of them, back to back. And the results were super interesting. I did this with a couple of different phones and a couple different scenarios. And what ended up surprising me, and quite a few other people, is it doesn't always look like the latest generation is always far and away the best. And I'll tell you why. See for years, smartphone cameras have been focused on getting better and better for regular photos, right? Because they were bad of course when it first started. The first iPhone camera was almost an afterthought. It was this little pinhole 2 megapixel webcam type of thing. No autofocus, no video recording. It didn't even have a selfie camera. But as people and culture valued the ability to take quick, high quality pictures and videos more and more, you of course see the cameras improving drastically on the back of our ph

In [ ]:
def Chunk_process(text,token_size,overlap):
  sentences = sent_tokenize(text)
  sent_token_counts = [len(decoder_tokenizer.encode(sent,add_special_tokens=False)) for sent in sentences]

  chunks = []
  current_chunk = []
  current_len = 0

  for sent,token_len in zip(sentences,sent_token_counts):
    if current_len + token_len <= token_size:
      current_chunk.append(sent) # because sent itself a senetence not character so no need to use " "
      current_len += token_len
    else:
      if current_chunk:
        chunks.append(" ".join(current_chunk))
      current_chunk = current_chunk[-overlap:] + [sent]
      current_len = token_len

  if current_chunk:
    chunks.append(" ".join(current_chunk))

  return chunks

In [ ]:
%%time
all_chunks = Chunk_process(transcript,token_size=400,overlap=1)
print(all_chunks[0])

 Alright, so I decided to try something. I broke out every single iPhone generation from 1 to 17 and took the same picture with every single one of them, back to back. And the results were super interesting. I did this with a couple of different phones and a couple different scenarios. And what ended up surprising me, and quite a few other people, is it doesn't always look like the latest generation is always far and away the best. And I'll tell you why. See for years, smartphone cameras have been focused on getting better and better for regular photos, right? Because they were bad of course when it first started. The first iPhone camera was almost an afterthought. It was this little pinhole 2 megapixel webcam type of thing. No autofocus, no video recording. It didn't even have a selfie camera. But as people and culture valued the ability to take quick, high quality pictures and videos more and more, you of course see the cameras improving drastically on the back of our phones year ove

In [ ]:
def summarizing_chunk(chunk):
  message = [
      {"role":"user","content":f"""Extract the key information from this transcript excerpt.
Keep the important explanations, definitions, and details the speaker mentioned.
Do not compress too much — preserve the meaning and explanations.

Excerpt:
{chunk}"""}
  ]
  prompt = decoder_tokenizer.apply_chat_template(message,tokenize=False,add_generation_prompt=True)

  inputs = decoder_tokenizer(prompt,return_tensors='pt').to(llm.device)

  with torch.no_grad():
    outputs = llm.generate(
        **inputs,
        max_new_tokens = 200,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        pad_token_id=decoder_tokenizer.eos_token_id
    )

  decoded = decoder_tokenizer.decode(outputs[0],skip_special_tokens=True) #outputs has [batch,sequence],skip_special_tokens: when we dont want unneccesory tokens in our output strings, set true
  return decoded[len(prompt):]

In [ ]:
def final_summarize(combined_summary):
  messages=[
      {'role':'system','content':'You are an expert YouTube video summarizer. Write in natural text like explaining the topic deeply.'},
      {"role": "user", "content": f"""Write a natural flowing summary of this video in 2-3 sentences only, starts with: "This video is about..." then explain what the speaker discussed. If the speaker used any specific instances, examples, instructions, researches, references , steps, guidelines or case studies, mention those seperately.
      Mention 4-5 key points with simple dashes (-) for bullet points. In the end write conclusion of the video with all the important words speaker used to describe. If speaker is mentioning any specific steps, installation guidelines or setup points, start them with "Steps: " keyword and then mention each one in bulletpoints
      Transcript summaries:{combined_summary}"""}
  ]

  prompt = decoder_tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
  inputs = decoder_tokenizer(prompt,return_tensors='pt').to(llm.device)
  input_length = inputs['input_ids'].shape[1]  # count only prompt tokens, so the ouput won't have prompts..

  with torch.no_grad():
    outputs = llm.generate(
        **inputs,
        max_new_tokens = 500,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        pad_token_id=decoder_tokenizer.eos_token_id
    )

  new_tokens = outputs[0][input_length:]
  decoded = decoder_tokenizer.decode(new_tokens,skip_special_tokens=True)
  return decoded.split("<|end_header_id|>")[-1].strip()

In [ ]:
%%time
chunk_summaries = [summarizing_chunk(c) for c in all_chunks]
combined = " ".join(chunk_summaries)
final = final_summarize(combined)
print(final)

This video is about a speaker comparing the camera performance of various iPhone generations, from iPhone 1 to iPhone 17, and finding that the latest generation is not always the best, contrary to the expectation that it would be. The speaker highlights the improvements in smartphone cameras over the years, driven by the increasing demand for high-quality camera capabilities, and notes that the difference between phones can be negligible in bright lighting conditions. The speaker emphasizes that the real difference between good and bad smartphone cameras lies in challenging conditions such as low light, fast-moving subjects, and deep zoom.

- The speaker compared two photos taken with the iPhone 17 and iPhone 11, which looked almost identical without zooming in or pixel peeping.
- The speaker notes that any phone from the past five years can take a "perfectly usable" photo in good lighting conditions, regardless of price.
- The speaker emphasizes that post-processing can degrade the ov